# Topic 22g - Writing the act report

A consolidation notebook. Everything here uses the whole Topic 22 toolkit and gives you no worked answers - only the checks that tell you whether your answer is right.

Work through the [main lesson](topic22_exploratory_data_analysis.ipynb) and notebooks 22a to 22f first.

## The scenario

The managing director liked the dashboard from Topic 21. Now she wants the written version - a one-page act report she can forward to the board without attaching a notebook. Five candidate findings are below, computed from angles the rest of this topic did not already use. Your job: rank them, pick the ones worth reporting, and write the report.

In [1]:
# Run this cell as it is.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_dir():
    """Locate the lesson's data folder from the repository root, the lesson
    folder, or the assignments folder, so this notebook runs from anywhere."""
    candidates = [
        Path.cwd() / "data",
        Path.cwd() / "exploratory_data_analysis" / "data",
        Path.cwd().parent / "data",
    ]
    for candidate in candidates:
        if (candidate / "air_quality_readings.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find air_quality_readings.csv. Looked in: " +
        ", ".join(str(c) for c in candidates)
    )


DATA_DIR = find_data_dir()

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#e1e0d9"
plt.rcParams["grid.linewidth"] = 0.8
plt.rcParams["axes.edgecolor"] = "#c3c2b7"
plt.rcParams["axes.axisbelow"] = True

# The course palette (Topic 21) - fixed hue order, never cycled, so a category
# keeps the same colour in every chart it appears in.
PALETTE = {
    "blue": "#2a78d6", "aqua": "#1baf7a", "yellow": "#eda100", "green": "#008300",
    "violet": "#4a3aa7", "red": "#e34948", "magenta": "#e87ba4", "orange": "#eb6834",
}
MUTED = "#c3c2b7"
INK = "#0b0b0b"
STATION_TYPE_COLOR = {
    "Roadside": PALETTE["red"], "Industrial": PALETTE["orange"],
    "Residential": PALETTE["aqua"], "Background": PALETTE["blue"],
}
STATION_TYPE_ORDER = ["Roadside", "Industrial", "Residential", "Background"]

WHO_ANNUAL_GUIDELINE = 5.0   # ug/m3, PM2.5 annual mean
WHO_24H_GUIDELINE = 15.0     # ug/m3, PM2.5 24-hour mean

print("pandas version:   ", pd.__version__)
print("data folder:      ", DATA_DIR)
print("files:            ", sorted(p.name for p in DATA_DIR.glob("*.csv")))

pandas version:    3.0.6
data folder:       /home/johnny/Projects/itfy-da/exploratory_data_analysis/data
files:             ['air_quality_readings.csv', 'air_quality_readings_assignment.csv', 'air_quality_stations.csv', 'air_quality_stations_assignment.csv', 'customers.csv', 'monthly_targets_wide.csv', 'orders.csv', 'products.csv', 'regions.csv', 'returns.csv']


In [2]:
# The retail "2023 Trading Review" table from Topic 20 - used for the two
# report-writing sections later in this notebook, since a wrangling report and
# an insights report have to be about work you actually did.
def clean_region(series):
    return (series.str.strip().str.replace(r"\s+", " ", regex=True).str.title())


def build_analysis_table(orders, customers, products, regions, returns):
    expected_rows = len(orders)
    customers = customers.copy()
    customers["region"] = clean_region(customers["region"])
    products = products.drop_duplicates(subset="product_id", keep="first")
    customers = customers.merge(regions, on="region", how="left", validate="m:1")
    table = (orders
             .merge(customers, on="customer_id", how="left", validate="m:1")
             .merge(products, on="product_id", how="left", validate="m:1"))
    returned_ids = set(returns["order_id"])
    table["was_returned"] = table["order_id"].isin(returned_ids)
    table["order_date"] = pd.to_datetime(table["order_date"])
    table["month"] = table["order_date"].dt.strftime("%b")
    table["revenue_ghs"] = (table["quantity"] * table["unit_price_ghs"]).round(2)
    table["cost_ghs"] = (table["quantity"] * table["unit_cost_ghs"]).round(2)
    table["profit_ghs"] = (table["revenue_ghs"] - table["cost_ghs"]).round(2)
    if len(table) != expected_rows:
        raise ValueError("Row count changed during wrangling.")
    return table.sort_values("order_date").reset_index(drop=True)


orders = pd.read_csv(DATA_DIR / "orders.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
regions = pd.read_csv(DATA_DIR / "regions.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")

analysis = build_analysis_table(orders, customers, products, regions, returns)
print(f"Retail analysis table: {analysis.shape[0]} rows x {analysis.shape[1]} columns")

Retail analysis table: 1200 rows x 23 columns


---

## Task 1 - Compute the five candidates

Run each of the five groupbys below and print the result. Do not interpret them yet.

1. Revenue by individual product (`product_name`), top 5
2. Return rate by individual product (`was_returned` mean x 100), sorted worst first, with each product's order count alongside
3. Revenue share by channel (percentage of total)
4. Order count by weekday
5. Monthly revenue by zone (the pivot you would need for a line chart)

In [3]:
# YOUR CODE HERE

## Task 2 - Apply the three checks

For each of the five candidates, apply the checks from 22e (sample size, shared cause, interesting vs. important) and decide: report it, or leave it out. Write one line per candidate.

(Hint: one of the five has a sample-size problem worth naming explicitly - a product with only 20-30 orders total can show a dramatic-looking return *rate* that is actually 7-9 returns.)

In [4]:
# YOUR CODE HERE
# decisions = {"candidate 1": "report" or "leave out", ...}  with a one-line reason each

## Task 3 - One chart per chosen insight

For each candidate you decided to report, build one chart - the right type for that question, per the chart-selection rule. Give each one a title that states the finding.

In [5]:
# YOUR CODE HERE

---

## Task 4 - Write the act report

Using the hook / ranked insights / so-what / limitations structure from the main lesson, write the full report as a Python string. Required:

- a one-sentence hook naming the overall headline number;
- 3-4 insights, each with at least one specific number;
- a "So what" paragraph naming at least one concrete action;
- a "Limitations" line.

In [6]:
act_report = """
YOUR REPORT HERE
"""

## Task 5 - Self-check your own report in code

Word count and section presence are things code can check, even though the writing quality itself cannot be. Run this cell against your `act_report`.

In [7]:
import re

word_count = len(act_report.split())
numbers_found = len(re.findall(r"\d[\d,]*\.?\d*", act_report))
has_so_what = "so what" in act_report.lower()
has_limitations = "limitation" in act_report.lower()

print(f"Word count: {word_count} (aim for roughly 150-350)")
print(f"Distinct numeric figures mentioned: {numbers_found} (aim for at least 5)")
print(f"Has a So What section: {has_so_what}")
print(f"Has a Limitations section: {has_limitations}")

Word count: 3 (aim for roughly 150-350)
Distinct numeric figures mentioned: 0 (aim for at least 5)
Has a So What section: False
Has a Limitations section: False


## Self-check

- [ ] All five candidates computed before any were judged - you did not skip straight to the ones that looked interesting
- [ ] The sample-size problem (a rate computed from very few orders) was caught and named, not reported as fact
- [ ] Every chosen insight has one chart, titled with the finding, not a generic label
- [ ] The report has a hook, 3-4 insights, a So What, and a Limitations line
- [ ] At least 5 distinct numbers appear in the report text
- [ ] No chart used is a pie or waffle with more than 5 slices
- [ ] No `# YOUR CODE HERE` remains anywhere

---
## Where this fits

- [Back to the main lesson](topic22_exploratory_data_analysis.ipynb)
- Previous: [22f - Writing the wrangling report](topic22f_writing_the_wrangling_report.ipynb)
- Next: [Topic 22 assignment](assignments/topic22_assignment_air_quality_eda.ipynb)